<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">

# **NOTEBOOK 13 — SOLUCIÓN**
---

## **Práctica 1: Uso de *embeddings* preentrenados (solución)**

Adaptamos el clasificador LSTM del notebook anterior para que use *embeddings* **GloVe** preentrenados en lugar de una capa de *embedding* que se inicializa al azar. En esta solución usamos los vectores de **100 dimensiones** (<code>glove.6B.100d.txt</code>). Para probar otro tamaño basta con cambiar la constante <code>GLOVE_DIM</code> (50, 100, 200 o 300).

Bibliotecas necesarias: <code>pip install torch datasets tokenizers huggingface_hub numpy</code>

#### **Dataset y tokenización**

Igual que en el notebook original: cargamos AG_NEWS con **datasets** y construimos el vocabulario a nivel de palabra con **tokenizers**. El tokenizador pasa el texto a minúsculas, y eso encaja con GloVe 6B, cuyo vocabulario también está en minúsculas.

In [ ]:
import time
import zipfile

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

GLOVE_DIM = 100  # 50, 100, 200 o 300

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Dispositivo:", device)

In [ ]:
dataset = load_dataset("fancyzhx/ag_news")
train_ds = dataset["train"]
test_ds = dataset["test"]

tokenizer = Tokenizer(models.WordLevel(unk_token="<unk>"))
tokenizer.normalizer = normalizers.Lowercase()
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.WordLevelTrainer(vocab_size=200_000, special_tokens=["<pad>", "<unk>"])
tokenizer.train_from_iterator(train_ds["text"], trainer=trainer)

PAD_IDX = tokenizer.token_to_id("<pad>")
VOCAB_SIZE = tokenizer.get_vocab_size()
print("Tamaño del vocabulario:", VOCAB_SIZE)

In [ ]:
text_pipeline = lambda x: tokenizer.encode(x).ids

def collate_batch(batch):
    label_list, text_list = [], []
    for sample in batch:
        text_list.append(torch.tensor(text_pipeline(sample["text"]), dtype=torch.long))
        label_list.append(sample["label"])
    return torch.tensor(label_list, dtype=torch.long), torch.nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)

train_dataloader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_batch)

#### **Descarga de los *embeddings* GloVe**

Descargamos <code>glove.6B.zip</code> (≈ 820 MB) del Hub de Hugging Face. <code>hf_hub_download</code> guarda el fichero en la caché local (<code>~/.cache/huggingface/hub</code>), así que solo se descarga la primera vez. No hace falta descomprimirlo en disco: con <code>zipfile</code> leemos directamente el fichero de texto que nos interesa.

In [ ]:
glove_zip = hf_hub_download(repo_id="stanfordnlp/glove", filename="glove.6B.zip")

with zipfile.ZipFile(glove_zip) as z:
    print(z.namelist())

#### **Matriz de *embeddings* alineada con nuestro vocabulario**

Una capa <code>nn.Embedding</code> es, en el fondo, una matriz de tamaño <code>(VOCAB_SIZE, embed_dim)</code> cuya fila <code>i</code> es el vector del token con índice <code>i</code>. Por eso hay que construir una matriz cuyas filas sigan **el orden de nuestro vocabulario** (el del tokenizador), copiando en cada fila el vector GloVe de esa palabra.

Los tokens que no aparecen en GloVe se tratan así:

- <code>&lt;pad&gt;</code>: vector de ceros, para que el relleno no aporte información.
- Palabras que no están en GloVe (incluido <code>&lt;unk&gt;</code>): vector aleatorio con la misma media y desviación típica que los vectores GloVe, para que tengan la misma escala que el resto.

In [ ]:
vocab = tokenizer.get_vocab()   # palabra -> índice
embedding_matrix = np.zeros((VOCAB_SIZE, GLOVE_DIM), dtype=np.float32)
found = np.zeros(VOCAB_SIZE, dtype=bool)

with zipfile.ZipFile(glove_zip) as z:
    with z.open(f"glove.6B.{GLOVE_DIM}d.txt") as f:
        for line in f:
            word, *values = line.decode("utf-8").rstrip().split(" ")
            idx = vocab.get(word)
            if idx is not None:
                embedding_matrix[idx] = np.asarray(values, dtype=np.float32)
                found[idx] = True

# Palabras sin vector GloVe: vector aleatorio con la estadística de GloVe
mean, std = embedding_matrix[found].mean(), embedding_matrix[found].std()
embedding_matrix[~found] = np.random.normal(mean, std, ((~found).sum(), GLOVE_DIM))
embedding_matrix[PAD_IDX] = 0.0

embedding_matrix = torch.from_numpy(embedding_matrix)
print(f"Palabras del vocabulario con vector GloVe: {found.sum()} de {VOCAB_SIZE} ({100 * found.mean():.1f} %)")

#### **Modelo LSTM con *embeddings* preentrenados**

El único cambio en el modelo es la capa de *embedding*. Usamos <code>nn.Embedding.from_pretrained</code>, que crea la capa a partir de una matriz ya existente:

- <code>freeze=True</code>: los vectores no se modifican durante el entrenamiento (se usan los de GloVe tal cual).
- <code>freeze=False</code>: se parte de GloVe y los vectores se siguen ajustando a nuestra tarea (*fine-tuning*).

La dimensión de entrada de la LSTM pasa a ser la de los vectores GloVe. El resto de la arquitectura no cambia.

In [ ]:
class LSTMTextClassificationModel(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_class, freeze=True):
        super(LSTMTextClassificationModel, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=freeze, padding_idx=PAD_IDX)  # <-- Embedding GloVe
        self.lstm = nn.LSTM(embedding_matrix.shape[1], hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_class)

    def forward(self, text):
        embedded = self.embedding(text)
        lstm_out, _ = self.lstm(embedded)
        # Tomar la última salida de la secuencia LSTM
        last_output = lstm_out[:, -1, :]
        output = self.fc(last_output)
        return output

In [ ]:
model = LSTMTextClassificationModel(embedding_matrix, 64, 4, freeze=True).to(device)

for batch in train_dataloader:
    predicted_label = model(batch[1].to(device))
    label = batch[0]
    break

print(batch[1][:4])
print(predicted_label[:4])
print(label[:4])

#### **Entrenamiento**

Las funciones <code>train</code> y <code>evaluate</code> son las del notebook original, con los mismos hiperparámetros, así que los resultados se pueden comparar directamente. Lo único nuevo es que los tensores se envían al dispositivo (<code>device</code>).

> Aunque el optimizador recibe todos los parámetros del modelo, con el *embedding* congelado sus pesos tienen <code>requires_grad=False</code> y SGD no los modifica.

In [ ]:
# Hyperparameters
EPOCHS = 10  # epoch
LR = 5  # learning rate

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR)

def train(dataloader):
    model.train()
    total_acc, total_count = 0, 0
    log_interval = 500

    for idx, (label, text) in enumerate(dataloader):
        label, text = label.to(device), text.to(device)
        optimizer.zero_grad()
        predicted_label = model(text)
        loss = criterion(predicted_label, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()

        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)
        if idx % log_interval == 0 and idx > 0:
            print('| {:5d} batches '
                  '| accuracy {:8.3f}'.format(idx, total_acc / total_count))
            total_acc, total_count = 0, 0


def evaluate(dataloader):
    model.eval()
    total_acc, total_count = 0, 0

    with torch.no_grad():
        for label, text in dataloader:
            label, text = label.to(device), text.to(device)
            predicted_label = model(text)
            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
    return total_acc / total_count

In [ ]:
for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()

    train(train_dataloader)
    accu_val = evaluate(test_dataloader)

    print("-" * 59)
    print(
        "| end of epoch {:3d} | time: {:5.2f}s | "
        "valid accuracy {:8.3f} ".format(
            epoch, time.time() - epoch_start_time, accu_val
        )
    )
    print("-" * 59)

#### **Preguntas de la práctica**

**¿Mejora la precisión?** Compara la precisión obtenida aquí con la del notebook original (embedding aleatorio de 32 dimensiones). Los vectores GloVe ya contienen información semántica aprendida de un corpus muy grande, así que lo normal es que el modelo aprenda más rápido (mejor precisión en las primeras épocas) y generalice mejor, sobre todo cuando hay pocos datos de entrenamiento. Además, con el *embedding* congelado el número de parámetros entrenables baja mucho (de unos 2 millones a unos 43.000 con GloVe 100d), porque la matriz de *embedding* ya no se entrena.

**¿Qué ocurre con distintos tamaños?** Cambia <code>GLOVE_DIM</code> y vuelve a ejecutar el notebook. Los vectores más grandes (200d y 300d) recogen más matices semánticos, pero la mejora suele ser pequeña a partir de 100d, mientras que el coste de memoria y de cómputo de la LSTM sí crece. Los de 50d son los más ligeros, aunque pueden quedarse algo cortos.

**Para seguir probando:** pon <code>freeze=False</code> para ajustar los vectores GloVe a la tarea (*fine-tuning*).